In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "query_id": [f"q{i}" for i in range(1000)],
    "intent": np.random.choice(["shopping", "inspiration", "navigational"], size=1000),
    "difficulty": np.random.choice(["easy", "medium", "hard"], size=1000),
    "score": np.random.uniform(1, 5, size=1000),
    "weight": np.random.uniform(0.1, 1.0, size=1000),
})

In [3]:
df

,query_id,intent,difficulty,score,weight
0,q0,shopping,hard,2.429275,0.529813
1,q1,shopping,easy,4.290688,0.427875
2,q2,navigational,medium,2.523914,0.870922
3,q3,shopping,easy,4.600409,0.622896
4,q4,inspiration,easy,2.649705,0.169837
...,...,...,...,...,...
995,q995,navigational,medium,2.466990,0.208601
996,q996,navigational,hard,2.883444,0.797672
997,q997,navigational,easy,3.926208,0.461496
998,q998,shopping,medium,4.955694,0.343854


## Simple Random Sampling

In [4]:
df.sample(n=100, random_state=80)

,query_id,intent,difficulty,score,weight
647,q647,shopping,hard,3.001725,0.201217
721,q721,shopping,hard,3.232181,0.769768
432,q432,shopping,medium,3.526791,0.399539
476,q476,navigational,easy,2.319295,0.672671
22,q22,inspiration,medium,4.662619,0.129476
...,...,...,...,...,...
512,q512,inspiration,hard,3.737230,0.385893
979,q979,inspiration,medium,1.764823,0.546084
426,q426,navigational,hard,2.550070,0.608192
772,q772,shopping,hard,3.919727,0.293170


## Systematic Sampling

In [6]:
step = 10

df.iloc[::step].reset_index(drop=True)

,query_id,intent,difficulty,score,weight
0,q0,shopping,hard,2.429275,0.529813
1,q10,navigational,medium,3.863057,0.812608
2,q20,navigational,easy,1.592247,0.735440
3,q30,inspiration,hard,3.003631,0.556984
4,q40,inspiration,hard,2.236359,0.985199
...,...,...,...,...,...
95,q950,navigational,easy,4.302958,0.566789
96,q960,inspiration,easy,2.287259,0.793287
97,q970,navigational,hard,4.207297,0.863208
98,q980,shopping,easy,1.125671,0.942761


## Bootstrap

In [7]:
df.sample(n=len(df), replace=True, random_state=42)

,query_id,intent,difficulty,score,weight
102,q102,inspiration,medium,1.375769,0.732011
435,q435,shopping,medium,4.110793,0.847182
860,q860,inspiration,hard,3.483552,0.739190
270,q270,navigational,medium,1.725688,0.992429
106,q106,navigational,hard,2.446647,0.354258
...,...,...,...,...,...
9,q9,inspiration,medium,1.502754,0.858850
823,q823,inspiration,hard,3.575940,0.468986
797,q797,navigational,easy,2.633955,0.196765
241,q241,navigational,medium,3.537155,0.473382


## Stratified Sampling (by one column)

In [8]:
def stratified_sample(df, stratify_col, n_per_group):
    return (
        df.groupby(stratify_col)
        .apply(lambda g: g.sample(n=min(len(g), n_per_group), random_state=42))
        .reset_index(drop=True)
    )

stratified_sample(df, "intent", n_per_group=50)

/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_10377/3951330520.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), n_per_group), random_state=42))


,query_id,intent,difficulty,score,weight
0,q678,inspiration,hard,1.549624,0.844270
1,q95,inspiration,hard,4.855081,0.338628
2,q743,inspiration,medium,3.304010,0.651829
3,q528,inspiration,easy,4.743330,0.277473
4,q125,inspiration,medium,2.682768,0.726458
...,...,...,...,...,...
145,q254,shopping,easy,2.175975,0.858237
146,q515,shopping,medium,4.386210,0.396878
147,q162,shopping,easy,3.438749,0.139699
148,q334,shopping,medium,3.508675,0.903079


## Strtified Sampling proportional to group sizes automatically

In [11]:
def proportional_sampling(df, group_cols, total_n, random_state=42):
    grouped = df.groupby(group_cols)
    group_sizes = grouped.size()

    # proportional to group size
    proportions = (group_sizes / group_sizes.sum()) * total_n
    proportions = proportions.round().astype(int)

    samples = []
    for group_key, group_df in grouped:
        n = min(proportions[group_key], len(group_df))
        samples.append(group_df.sample(n=n, replace=False, random_state=random_state))

    # return pd.concat(samples, axis=0).reset_index(drop=True)
    return pd.concat(samples, axis=0)

In [12]:
proportional_sampling(df, ["intent", "difficulty"], total_n=100)

,query_id,intent,difficulty,score,weight
477,q477,inspiration,easy,4.348803,0.821420
719,q719,inspiration,easy,4.037558,0.169954
25,q25,inspiration,easy,1.224142,0.610007
376,q376,inspiration,easy,4.237415,0.214968
150,q150,inspiration,easy,2.179152,0.678908
...,...,...,...,...,...
397,q397,shopping,medium,2.967794,0.928597
310,q310,shopping,medium,3.899964,0.805230
439,q439,shopping,medium,1.688241,0.368105
155,q155,shopping,medium,3.964059,0.154948


## Weighted Sampling

In [13]:
weights = df["weight"] / df["weight"].sum()

df.sample(n=100, weights=weights, random_state=42)

,query_id,intent,difficulty,score,weight
369,q369,shopping,medium,2.469991,0.614723
951,q951,inspiration,easy,2.623957,0.713805
733,q733,shopping,medium,4.863875,0.760503
592,q592,navigational,easy,2.958217,0.328074
153,q153,inspiration,medium,2.639243,0.659171
...,...,...,...,...,...
106,q106,navigational,hard,2.446647,0.354258
33,q33,inspiration,hard,4.849617,0.736292
635,q635,shopping,hard,2.629235,0.441044
316,q316,navigational,hard,1.547716,0.704625


## Neyman Allocation Stratified Sampling

Optimized sample size per group based on variance (best statistically).

In [14]:
def neyman_allocation(df, stratify_col, total_n):
    groups = df.groupby(stratify_col)
    Nh = groups.size()
    Sh = groups["score"].std()

    # weights
    alloc = (Nh * Sh) / (Nh * Sh).sum() * total_n
    alloc = alloc.round().astype(int)

    # sample from each stratum
    samples = [
        groups.get_group(h).sample(n=int(alloc[h]), replace=False, random_state=42)
        for h in alloc.index
    ]
    return pd.concat(samples).reset_index(drop=True)

In [15]:
neyman_allocation(df, "difficulty", total_n=150)

,query_id,intent,difficulty,score,weight
0,q675,shopping,easy,3.392545,0.560055
1,q320,shopping,easy,4.018478,0.951694
2,q307,navigational,easy,2.802191,0.666104
3,q99,inspiration,easy,4.800326,0.728986
4,q341,inspiration,easy,1.560477,0.592771
...,...,...,...,...,...
145,q884,navigational,medium,4.811132,0.525270
146,q867,inspiration,medium,3.915693,0.389365
147,q166,shopping,medium,3.079568,0.551718
148,q8,shopping,medium,3.530838,0.251958


## Cluster Sampling (sample clusters, not rows)

In [16]:
clusters = df["intent"].unique()
chosen_clusters = np.random.choice(clusters, size=2, replace=False)

In [17]:
df[df["intent"].isin(chosen_clusters)]

,query_id,intent,difficulty,score,weight
2,q2,navigational,medium,2.523914,0.870922
4,q4,inspiration,easy,2.649705,0.169837
6,q6,navigational,medium,2.218286,0.443895
7,q7,inspiration,easy,4.072780,0.458765
9,q9,inspiration,medium,1.502754,0.858850
...,...,...,...,...,...
993,q993,inspiration,medium,2.500875,0.223863
994,q994,inspiration,medium,1.280717,0.485320
995,q995,navigational,medium,2.466990,0.208601
996,q996,navigational,hard,2.883444,0.797672


## Quota Sampling (Ensure exact counts by category)

In [19]:
quota = {
    "shopping": 30,
    "inspiration": 40,
    "navigational": 30
}

pd.concat([
    df[df["intent"] == key].sample(n=value, random_state=42)
    for key, value in quota.items()
])

,query_id,intent,difficulty,score,weight
105,q105,shopping,easy,2.347020,0.185971
956,q956,shopping,easy,2.109918,0.340305
255,q255,shopping,easy,4.881934,0.747895
554,q554,shopping,hard,3.525421,0.622749
200,q200,shopping,medium,2.650264,0.863181
...,...,...,...,...,...
249,q249,navigational,hard,3.309321,0.484917
503,q503,navigational,medium,2.806811,0.116797
530,q530,navigational,hard,2.299521,0.281844
208,q208,navigational,hard,3.837248,0.446559


## Probability Proportional-to-Size (PPS) Sampling

In [20]:
probs = df["score"] / df["score"].sum()
df.sample(n=100, weights=probs, random_state=42)

,query_id,intent,difficulty,score,weight
378,q378,shopping,medium,2.435217,0.969183
951,q951,inspiration,easy,2.623957,0.713805
737,q737,inspiration,medium,3.971608,0.120413
597,q597,shopping,hard,2.281909,0.304101
155,q155,shopping,medium,3.964059,0.154948
...,...,...,...,...,...
108,q108,shopping,hard,2.150638,0.530708
29,q29,navigational,easy,4.908090,0.467310
638,q638,inspiration,easy,3.934530,0.875700
324,q324,inspiration,medium,3.080816,0.752908
